In [13]:
import pandas as pd
import pulp
energy = energy = pd.read_parquet('../Data/cleaned_data/dam.parquet')   #change to model produced schedule
prices = energy['lmp'].iloc[:288].to_numpy()                            #numaric array of prices
n = len(prices)

In [14]:
#creating the optimization model
model = pulp.LpProblem(
    "MISO_Optimization",    #name of optimaztion model
    pulp.LpMaximize)        #we are trying to find the max profit in our model

In [15]:
#creating decision variables
buy = model.add_variable_dicts( #creating a dictionary of decision variables
    'Buy',                  #name of each indices of decision variables
    range(n),               #represents the indices
    cat='Binary')           #Variable is either 1 or 0
sell = model.add_variable_dicts('Sell', range(n), cat='Binary')
battery = model.add_variable_dicts('Battery', range(n), cat='Binary')

In [16]:
#objective
model += pulp.lpSum(        #adding multiple expressions together
    prices[i] * sell[i] - prices[i] * buy[i]    #profit equation
    for i in range(n)       #for each interval
)

In [17]:

for i in range(n):
    #Battery starts full
    if i == 0:
        model += battery[i] == 1 + buy[i] - sell[i]
    else:
        model += battery[i] == battery[i-1] + buy[i] - sell[i]

    #cannot buy and sell simultaneously
    model += buy[i] + sell[i] <= 1

In [23]:
#solving the modoel
stats = model.solve(
    pulp.COIN_CMD(msg=False)) #selects solver

#results
print("Status:", stats.status_str)
if stats.status == pulp.LpSolveStatus.Optimal:
    for i in range(n):
        print(
            "Interval:", i + 1,
            "Buy:", pulp.value(buy[i]),
            "Sell:", pulp.value(sell[i]),
            "Battery:", pulp.value(battery[i])
        )

Status: Optimal
Interval: 1 Buy: 0.0 Sell: 1.0 Battery: 0.0
Interval: 2 Buy: 1.0 Sell: 0.0 Battery: 1.0
Interval: 3 Buy: 0.0 Sell: 1.0 Battery: 0.0
Interval: 4 Buy: 0.0 Sell: 0.0 Battery: 0.0
Interval: 5 Buy: 0.0 Sell: 0.0 Battery: 0.0
Interval: 6 Buy: 0.0 Sell: 0.0 Battery: 0.0
Interval: 7 Buy: 0.0 Sell: 0.0 Battery: 0.0
Interval: 8 Buy: 1.0 Sell: 0.0 Battery: 1.0
Interval: 9 Buy: 0.0 Sell: 0.0 Battery: 1.0
Interval: 10 Buy: 0.0 Sell: 0.0 Battery: 1.0
Interval: 11 Buy: 0.0 Sell: 1.0 Battery: 0.0
Interval: 12 Buy: 1.0 Sell: 0.0 Battery: 1.0
Interval: 13 Buy: 0.0 Sell: 1.0 Battery: 0.0
Interval: 14 Buy: 0.0 Sell: 0.0 Battery: 0.0
Interval: 15 Buy: 0.0 Sell: 0.0 Battery: 0.0
Interval: 16 Buy: 1.0 Sell: 0.0 Battery: 1.0
Interval: 17 Buy: 0.0 Sell: 1.0 Battery: 0.0
Interval: 18 Buy: 0.0 Sell: 0.0 Battery: 0.0
Interval: 19 Buy: 0.0 Sell: 0.0 Battery: 0.0
Interval: 20 Buy: 1.0 Sell: 0.0 Battery: 1.0
Interval: 21 Buy: 0.0 Sell: 0.0 Battery: 1.0
Interval: 22 Buy: 0.0 Sell: 0.0 Battery: 1.0
Int

In [30]:
#track results
profit = 0
buys = 0
sells = 0
history = []

#only process results if optimization succeeded
if stats.status == pulp.LpSolveStatus.Optimal:
    for i in range(n):
        current_price = prices[i]

        #check if PuLP chose to buy
        if pulp.value(buy[i]) > 0.5:
            action = 'Buy'
            profit -= current_price
            buys += 1

        #check if Pulp chose to sell
        elif pulp.value(sell[i]) > 0.5:
            action = 'Sell'
            profit += current_price
            sells += 1
        else:
            action = 'Hold'

        #get battery state after trading
        battery_state = round(pulp.value(battery[i]))

        #save results for this interval
        history.append({
            'interval':i+1,
            'Current Price': current_price,
            'Next Price': prices[i+1] if i < n-1 else None,
            'Action': action,
            'Battery': 'Full' if battery_state == 1 else 'Empty',
            'Profit': round(profit, 2)
        })
    history = pd.DataFrame(history)

    print('\nResults')
    print('Total Profit $', round(profit, 2))
    print('Total Buys:', buys)
    print('Total Sells:', sells)

    display(history.head(20)) 
else:
    print('Optimizatioin did not return an optimal solution')   


Results
Total Profit $ 302.87
Total Buys: 47
Total Sells: 48


,interval,Current Price,Next Price,Action,Battery,Profit
0,1,26.92,23.70,Sell,Empty,26.92
1,2,23.70,23.86,Buy,Full,3.22
2,3,23.86,23.62,Sell,Empty,27.08
3,4,23.62,23.50,Hold,Empty,27.08
4,5,23.50,22.21,Hold,Empty,27.08
5,6,22.21,21.88,Hold,Empty,27.08
6,7,21.88,21.20,Hold,Empty,27.08
7,8,21.20,21.56,Buy,Full,5.88
8,9,21.56,23.57,Hold,Full,5.88
9,10,23.57,24.22,Hold,Full,5.88


In [31]:
history.to_parquet('lp_results.parquet', index=False)